<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Own_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -------------------------------------------------
#  Module 11
# -------------------------------------------------

# 1. Manual Logistic Regression from Scratch

## Dataset: Breast Cancer Wisconsin Dataset

Download:
•	https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

Tasks
- 1.	Load the dataset and drop any ID / unnamed columns.
- 2.	Split it into 80% training and 20% testing data.
- 3.	Scale the features with StandardScaler and encode the target with LabelEncoder.
- 4.	Convert the data to PyTorch tensors.
- 5.	Implement a single-neuron model by hand: random weights and zero bias (both with requires_grad=True), a sigmoid forward pass, and a binary cross-entropy loss function (no torch.nn, no torch.optim).
- 6.	Train for 25 epochs with a manual gradient-descent update loop, printing the loss every epoch.
- 7.	Evaluate accuracy on the test set.
Does the loss decrease smoothly across all 25 epochs? If not, what might be causing the jumps?


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load the dataset and drop any ID / unnamed columns.
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

X = df.drop(['id', 'diagnosis', 'Unnamed: 32'], axis = 1)
y = df['diagnosis']

# Split it into 80% training and 20% testing data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scale the features with StandardScaler and encode the target with LabelEncoder.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Convert the data to PyTorch tensors.
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).reshape(-1, 1)
X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).reshape(-1, 1)

# Implement a single-neuron model by hand: random weights and zero bias (both with requires_grad=True), a sigmoid forward pass, and a binary cross-entropy loss function (no torch.nn, no torch.optim).
class SimpleModel():
  def __init__(self, x):
    self.weights = torch.rand(x.shape[1], 1, dtype = torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)

  def forward(self, x):
    z = torch.matmul(x, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_func(self, y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    loss = -torch.mean(y * torch.log(y_pred) + (1 - y) * (torch.log(1 - y_pred)))
    return loss


# Train for 25 epochs with a manual gradient-descent update loop, printing the loss every epoch.
learning_rate = 0.1
epochs = 25

model = SimpleModel(X_train_tensor)

for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculation
  loss = model.loss_func(y_pred, y_train_tensor)

  # backpropagation
  loss.backward()

  # update weights and bias
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in epochs
  print(f'epochs: {epoch + 1}, loss: {loss.item()}')


print()
print()
print()

# Evaluate accuracy on the test set. Does the loss decrease smoothly across all 25 epochs? If not, what might be causing the jumps?
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'accuracy: {accuracy.item()}')


# In plain terms:

# The loss should go down smoothly, step by step, without any weird jumps back up — and it did.
# That's because logistic regression's loss has a single bowl-shaped curve (convex) — there's only one minimum, so gradient descent just walks straight downhill toward it, no zigzagging between different "valleys."
# You're also using all the training data at once each step (not small random batches), so there's no randomness making the loss bounce around epoch to epoch.

# If you ever did see the loss jump around instead of decreasing steadily, it would usually mean one of these:

# Learning rate too high — you're taking steps so big you overshoot past the minimum and bounce around it instead of settling in.
# Features not scaled properly — if one feature has huge values compared to others, it dominates the gradient and throws off the updates.
# A code bug, like the shape mismatch earlier — feeding the model bad-shaped data corrupts the gradient calculation and gives you garbage updates.

epochs: 1, loss: 0.5226084517090123
epochs: 2, loss: 0.5075175172042845
epochs: 3, loss: 0.492823537472651
epochs: 4, loss: 0.4785245835193676
epochs: 5, loss: 0.4646204962914867
epochs: 6, loss: 0.4511125004054087
epochs: 7, loss: 0.4380025250284362
epochs: 8, loss: 0.4252922422081844
epochs: 9, loss: 0.41298195226126017
epochs: 10, loss: 0.4010695649403179
epochs: 11, loss: 0.38954997063911667
epochs: 12, loss: 0.37841501250164233
epochs: 13, loss: 0.36765407769923875
epochs: 14, loss: 0.357255123492924
epochs: 15, loss: 0.34666324096732914
epochs: 16, loss: 0.3346876679567407
epochs: 17, loss: 0.3231089950286521
epochs: 18, loss: 0.31191886647698097
epochs: 19, loss: 0.3011116598402604
epochs: 20, loss: 0.290684060035178
epochs: 21, loss: 0.28063433516050407
epochs: 22, loss: 0.27096148447931473
epochs: 23, loss: 0.26166441746780345
epochs: 24, loss: 0.252741294167334
epochs: 25, loss: 0.24418910822308337



accuracy: 0.9649122953414917


In [1]:
# -------------------------------------------------
#  Module 12
# -------------------------------------------------

In [15]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


# Train test split

In [9]:
X = df.drop(['id', 'diagnosis', 'Unnamed: 32'], axis = 1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Scale and Encoding

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Numpy array to tensor

In [14]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

# Change dtypes
X_train_tensor = torch.tensor(X_train_tensor, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_tensor, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train_tensor, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test_tensor, dtype = torch.float32)

/tmp/ipykernel_664/1947649039.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_tensor = torch.tensor(X_train_tensor, dtype = torch.float32)
/tmp/ipykernel_664/1947649039.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_tensor = torch.tensor(X_test_tensor, dtype = torch.float32)
/tmp/ipykernel_664/1947649039.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_tensor = torch.tensor(y_train_tensor, dtype = torch.float32)
/tmp/ipykernel_664/1947649039.py:10: UserWarning: To copy construct from a tensor, it is reco

# Define model

In [16]:
class SimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

# Training pipeline

In [21]:
learning_rate = 0.1
epochs = 25

# define loss function
loss_function = nn.BCELoss()

In [22]:
# create model
model = SimpleNN(X_train_tensor.shape[1])

# optimizer
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

# define loop
for epoch in range(epochs):
  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculation
  loss = loss_function(y_pred, y_train_tensor.reshape(-1, 1))

  # zero gradient
  optimizer.zero_grad()

  # backpropagation
  loss.backward()

  # parameters update
  optimizer.step()

  # print loss and epochs
  print(f'epochs: {epoch + 1}, loss: {loss.item()}')

epochs: 1, loss: 0.48943257331848145
epochs: 2, loss: 0.4110720455646515
epochs: 3, loss: 0.36321601271629333
epochs: 4, loss: 0.33021485805511475
epochs: 5, loss: 0.30567634105682373
epochs: 6, loss: 0.2864938974380493
epochs: 7, loss: 0.27095451951026917
epochs: 8, loss: 0.2580263018608093
epochs: 9, loss: 0.2470458298921585
epochs: 10, loss: 0.23756463825702667
epochs: 11, loss: 0.22926729917526245
epochs: 12, loss: 0.22192464768886566
epochs: 13, loss: 0.21536563336849213
epochs: 14, loss: 0.2094595581293106
epochs: 15, loss: 0.20410460233688354
epochs: 16, loss: 0.19922004640102386
epochs: 17, loss: 0.19474093616008759
epochs: 18, loss: 0.19061435759067535
epochs: 19, loss: 0.18679654598236084
epochs: 20, loss: 0.18325117230415344
epochs: 21, loss: 0.17994748055934906
epochs: 22, loss: 0.17685949802398682
epochs: 23, loss: 0.1739649474620819
epochs: 24, loss: 0.17124469578266144
epochs: 25, loss: 0.1686820685863495


# Model evaluation

In [23]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5301631093025208


# Summary

In [24]:
X_train_tensor.shape

torch.Size([455, 30])

In [27]:
!pip install torchinfo

In [28]:
from torchinfo import summary

In [29]:
summary(model, input_size = (455, 30))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleNN                                 [455, 1]                  --
├─Linear: 1-1                            [455, 1]                  31
├─Sigmoid: 1-2                           [455, 1]                  --
Total params: 31
Trainable params: 31
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.05
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.06